In [10]:
import json, functools, getpass, os
from openai import OpenAI, RateLimitError, APIConnectionError
from google.colab import userdata

# Initialize API key securely
api_key = os.environ.get('GROQ_API_KEY')

if not api_key:
    try:
        # Try getting it from Colab secrets
        api_key = userdata.get('GROQ_API_KEY')
    except Exception:
        pass

if not api_key:
    try:
        api_key = userdata.get('OPENAI_API_KEY')
    except Exception:
        pass

if not api_key:
    print("API Key not found in environment or Colab secrets.")
    api_key = getpass.getpass("Please paste your API Key and press Enter: ")

if not api_key:
    raise ValueError("An API key is required to initialize the client.")

# Configure client base URL based on key type (Groq vs OpenAI)
if api_key.startswith("gsk_"):
    print("Groq API key detected. Routing requests to Groq endpoint.")
    client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")
else:
    print("OpenAI API key detected. Routing requests to default OpenAI endpoint.")
    client = OpenAI(api_key=api_key)

MODEL = "openai/gpt-oss-120b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    try:
        text = input("You: ")
        if text.lower().strip() in ("bye", "exit", "quit"):
            print("Bot: Thank you for visiting Sweet Crumbs!")
            break
        print("Bot:", chat(text))
    except KeyboardInterrupt:
        print("\nChat session interrupted. Goodbye!")
        break

Groq API key detected. Routing requests to Groq endpoint.
You: hi
Bot: Hi! 😊 Welcome to Sweet Crumbs Home Bakery. How can I help you today? We have plum cake (₹500), chocolate cake (₹650) and cupcakes (₹80). Let me know what you’d like!
You: how are u
Bot: I’m doing great, thank you! 😊 How can I assist you with our cakes or cupcakes today?
You: ok
Bot: Great! Would you like to place an order or know more about our plum cake, chocolate cake, or cupcakes? Let me know your choice and any special requests (eggless available). I’m here to help!
You: order
Bot: Sure! May I have your name, the items (plum cake, chocolate cake, cupcakes), quantities and if you’d like any egg‑less versions? I’ll note it down for you.
You: sai
Bot: Nice to meet you, Sai! 🌟 Please tell me which items (plum cake, chocolate cake, cupcakes) and quantities you’d like, and if any should be egg‑less. I’ll note it down for you.
You: creame cake
Bot: I’ll check with the owner.
You: ok check 
Bot: I'll check with the owne